# Exploring Hamilton DAGs with iacs Data

Hamilton is a framework for defining dataflows (DAGs) using plain Python functions:
- **Function names** become node names
- **Parameter names** declare dependencies on other nodes
- The DAG is constructed lazily — nothing executes until you call `driver.execute()`

Here we define a small Hamilton dataflow over iacs registry tables and visualize the DAG.

In [ ]:
!uv pip install sf-hamilton[visualization]

In [ ]:
import ibis
from hamilton import driver
from iacs.io_system import IOSystem
from iacs.registry import Registry

ibis.options.interactive = True

io = IOSystem()
entity_centered = io.read_entity_centered_file("../components/components.yaml")
registry = Registry.from_entity_centered(entity_centered)
print(f"Component types: {registry.component_types}")

## Define a Hamilton dataflow module

Hamilton normally loads functions from a `.py` module. We can emulate this by
attaching functions to a `types.ModuleType` object and passing it to the driver.

In [ ]:
import types
import ibis

dataflow = types.ModuleType("dataflow")

def id_table(registry: Registry) -> ibis.Table:
    return registry.table("id")

def description_table(registry: Registry) -> ibis.Table:
    return registry.table("description")

def parent_table(registry: Registry) -> ibis.Table:
    return registry.table("parent")

def entity_summary(id_table: ibis.Table, description_table: ibis.Table) -> ibis.Table:
    """Join id and description to get readable entity summaries."""
    return id_table.join(
        description_table, "entity_id"
    ).select("entity_id", "path", "key", description="value_right")

def root_entities(entity_summary: ibis.Table, parent_table: ibis.Table) -> ibis.Table:
    """Find entities that have no parent (root-level entities)."""
    has_parent = parent_table.select("entity_id").distinct()
    return entity_summary.anti_join(has_parent, "entity_id")

def entity_child_count(parent_table: ibis.Table) -> ibis.Table:
    """Count the number of children per parent entity."""
    return parent_table.group_by("value").aggregate(
        child_count=parent_table.entity_id.count()
    ).rename(entity_id="value")

# Attach all functions to the module
for name, obj in list(locals().items()):
    if callable(obj) and not name.startswith("_"):
        setattr(dataflow, name, obj)

In [ ]:
dr = driver.Driver({"registry": registry}, dataflow)
dr.display_all_functions()

## Execute the DAG

In [ ]:
result = dr.execute(["entity_summary", "root_entities", "entity_child_count"])

In [ ]:
result["entity_summary"]

In [ ]:
result["root_entities"]

In [ ]:
result["entity_child_count"]

## Visualize execution

Show which nodes were traversed to compute `root_entities`.

In [ ]:
dr.visualize_execution(["root_entities"], "./hamilton_execution.png", {"format": "png"})